# ParkSense · diffusion image-to-image weather translation (free Colab GPU)

This notebook is the **diffusion half** of the ParkSense generative-AI component (the CycleGAN half runs on CPU, see `training/cyclegan.py`).

It uses **InstructPix2Pix** (Brooks et al., CVPR 2023; open weights `timbrooks/instruct-pix2pix`, MIT-style licence) to turn dry daytime frames into **night, fog and rain** versions while keeping the scene geometry, so the original bay polygons and labels still apply.

Outputs are used two ways:
1. **Stress test**: translate the UFPR04 *test* frames and measure how much accuracy drops (night and fog have no real labelled data in PKLot).
2. **Augmentation**: translate the UFPR04 *training* frames, retrain the CNN with them, and measure the effect on real rain and on the unseen car parks.

Runtime → Change runtime type → **T4 GPU** (free tier). Full run: about 40–60 minutes.

In [ ]:
import os
if not os.path.exists("parksense"):
    !git clone -q https://github.com/adarsh836/parksense.git
%cd /content/parksense
!pip -q install diffusers accelerate

## 1 · Data: the same manifest as the CPU pipeline (MetaPKLot images via git sparse checkout)

In [ ]:
!bash training/fetch_data.sh        # annotations + the sampled frames (~300 MB)
!python training/extract_crops.py

## 2 · Translate frames with InstructPix2Pix

In [ ]:
import csv, os, torch, numpy as np, cv2
from PIL import Image
from diffusers import StableDiffusionInstructPix2PixPipeline, EulerAncestralDiscreteScheduler

pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(
    "timbrooks/instruct-pix2pix", torch_dtype=torch.float16, safety_checker=None).to("cuda")
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.set_progress_bar_config(disable=True)

PROMPTS = {
    "night": "make it night time, dark sky, street lights, photographed by a security camera",
    "fog":   "make it a thick foggy morning",
    "rain":  "make it raining heavily, wet asphalt with reflections",
}
man = list(csv.DictReader(open("data/manifest.csv")))

def translate(path, prompt, seed=0):
    img = cv2.imread(os.path.join("data/images", path))
    H, W = img.shape[:2]
    small = cv2.resize(img, (768, 432), interpolation=cv2.INTER_AREA)
    g = torch.Generator("cuda").manual_seed(seed)
    out = pipe(prompt, image=Image.fromarray(cv2.cvtColor(small, cv2.COLOR_BGR2RGB)),
               num_inference_steps=20, image_guidance_scale=1.6, guidance_scale=7.0, generator=g).images[0]
    out = cv2.cvtColor(np.array(out), cv2.COLOR_RGB2BGR)
    up = cv2.resize(out, (W, H), interpolation=cv2.INTER_CUBIC).astype(np.float32)
    # Laplacian detail transfer: keep the original high-frequency structure (bay lines, car edges)
    detail = img.astype(np.float32) - cv2.resize(small, (W, H), interpolation=cv2.INTER_CUBIC).astype(np.float32)
    return np.clip(up + 0.6 * detail, 0, 255).astype(np.uint8)

def run(rows, cond, root):
    for k, r in enumerate(rows):
        dst = os.path.join(root, cond, r["file"]); os.makedirs(os.path.dirname(dst), exist_ok=True)
        if not os.path.exists(dst):
            cv2.imwrite(dst, translate(r["file"], PROMPTS[cond], seed=k), [cv2.IMWRITE_JPEG_QUALITY, 92])

test_dry = [r for r in man if r["site"] == "UFPR04" and r["split"] == "test" and r["weather"] != "Rainy"]
train = [r for r in man if r["site"] == "UFPR04" and r["split"] == "train"]
import time
t0 = time.time()
for cond in PROMPTS:
    run(test_dry, cond, "data/stress_diffusion")      # 90 frames per condition
    run(train[::2], cond, "data/gen_diffusion")        # ~100 frames per condition (augmentation)
    print(cond, "done", round((time.time() - t0) / 60, 1), "min")
# a small gallery for the report
import numpy as np
rows = []
for r in test_dry[:3]:
    row = [cv2.resize(cv2.imread(os.path.join("data/images", r["file"])), (480, 270))]
    for cond in PROMPTS:
        row.append(cv2.resize(cv2.imread(os.path.join("data/stress_diffusion", cond, r["file"])), (480, 270)))
    rows.append(np.hstack(row))
os.makedirs("results/gallery", exist_ok=True)
cv2.imwrite("results/gallery/diffusion_examples.jpg", np.vstack(rows), [cv2.IMWRITE_JPEG_QUALITY, 85])

Look at a few results before using them: InstructPix2Pix sometimes moves or removes cars. The check below measures **label preservation** (does the CPU-trained CNN still agree with the original labels on translated *dry→rain* frames?) and you should also inspect frames by eye.

In [ ]:
import glob, matplotlib.pyplot as plt
fs = sorted(glob.glob("data/stress_diffusion/*/PKLot/UFPR04/*/*/*.jpg"))[:6]
plt.figure(figsize=(16, 9))
for i, f in enumerate(fs):
    plt.subplot(2, 3, i + 1); plt.imshow(cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.title(f.split("/")[2])

## 3 · Crops, stress test and augmentation experiment

In [ ]:
for cond in ["night", "fog", "rain"]:
    !python training/gen_crops.py --frames data/stress_diffusion/{cond} --out data/stress_diffusion_{cond}.npz
    !python training/gen_crops.py --frames data/gen_diffusion/{cond}    --out data/gen_diffusion_{cond}.npz
# augmentation arm: same recipe as the deployed model (all 3 PKLot parks + photometric) plus diffusion frames
!python training/extract_crops.py --splits train_multi --out data/crops_multi.npz
!python training/train_cnn.py --name cnn_multi_diffusion_s0 --seed 0 --epochs 14 --aug photometric --train_extra data/crops_multi.npz --extra data/gen_diffusion_night.npz data/gen_diffusion_fog.npz data/gen_diffusion_rain.npz --extra_frac 0.3
# protocol arm E: UFPR04 only + diffusion frames (compare with arms A-C)
for s in range(3):
    !python training/train_cnn.py --name cnn_diffusion_s{s} --seed {s} --epochs 14 --extra data/gen_diffusion_night.npz data/gen_diffusion_fog.npz data/gen_diffusion_rain.npz --extra_frac 0.3
# stress test of every model on the diffusion frames (independent of the photometric simulation)
!python training/stress_test.py --sets data/stress_cyclegan_rain.npz data/stress_diffusion_night.npz data/stress_diffusion_fog.npz data/stress_diffusion_rain.npz
!python training/summarise.py > /dev/null

In [ ]:
# Print everything needed for the report, and pack the results for download
print(open("results/stress.md").read())
import json, glob
for f in sorted(glob.glob("runs/cnn_*diffusion*/meta.json")):
    m = json.load(open(f)); print(m["name"], {k: round(v["acc"], 4) for k, v in m["pooled"].items()})
!zip -qr /content/diffusion_results.zip results runs/cnn_*diffusion*/meta.json runs/cnn_*diffusion*/model.pt
print("RESULTS READY")